In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
N = 100 # Gear_Ratio

folder_path = r"S:/Nit Durgapur/College 6th Sem/CSIR-CMERI/Friction_Modelling/EXP_1/EXP_1_Data/12_06_2026"
file = r"J_1_12.06.2026_13_05_36.csv"

file_path = os.path.join(folder_path, file)
df = pd.read_csv(file_path)
print(df.shape)
df.head()

In [ ]:
df["q"] = np.deg2rad(df["q1"])
df["v"] = np.deg2rad(df["v1"])
df["tm"] = df["m_t_1"]
df["tj"] = df["jts1"]

In [ ]:
df["tf"] = (N * df["tm"]) - df["tj"]

In [ ]:
size = len(df)

train_end = int(0.60 * size)
val_end = int(0.80 * size)

train_df = df.iloc[: train_end]
val_df = df.iloc[train_end : val_end]
test_df = df.iloc[val_end :]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
def model1(v, Tc, Bv, Vs):
    return (Tc * np.tanh(v/Vs) + Bv * v)

In [ ]:
v_train = train_df["v"].values
tf_train = train_df["tf"].values

p0 = [5, 0.1, 0.01]

params1, _ = curve_fit(
    model1,
    v_train,
    tf_train,
    p0 = p0,
    maxfev = 50000
)

Tc1, Bv1, Vs1 = params1

print("Tc =",Tc1)
print("Bv =",Bv1)
print("Vs =",Vs1)

In [ ]:
v_test = test_df["v"].values
tf_test = test_df["tf"].values

pred1 = model1(v_test, * params1)

In [ ]:
rmse1 = np.sqrt(mean_squared_error(tf_test, pred1))
mae1 = mean_absolute_error(tf_test, pred1)
r2_1 = r2_score(tf_test, pred1)

print(rmse1)
print(mae1)
print(r2_1)

In [ ]:
def model2(v, Ts, Tc, Bv, Vs, alpha):
    return ((Tc + (Ts-Tc) * np.exp(-(np.abs(v)/Vs) ** alpha)) * np.tanh(v/0.001) + Bv * v)

In [ ]:
p0 = [10, 5, 0.1, 0.05, 1]

params2,_ = curve_fit(
    model2,
    v_train,
    tf_train,
    p0=p0,
    maxfev=100000
)

Ts, Tc, Bv, Vs, alpha = params2

print("Tc = ", Tc)
print("Ts = ", Ts)
print("Bv = ", Bv)
print("Vs = ", Bv)
print("alpha = ", alpha)

In [ ]:
pred2 = model2(v_test, * params2)

In [ ]:
rmse2 = np.sqrt(mean_squared_error(tf_test, pred2))
mae2 = mean_absolute_error(tf_test, pred2)
r2_2 = r2_score(tf_test, pred2)

print(rmse2)
print(mae2)
print(r2_2)

In [ ]:
pred_train_stribeck = model2(v_train, * params2)

residual_train = (tf_train - pred_train_stribeck)

In [ ]:
features = ["q", "v", "tm", "tj"]

X_train = train_df[features].values
X_val = val_df[features].values
X_test = test_df[features].values

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
pred_val_stribeck = model2(val_df["v"].values, * params2)

residual_val = (val_df["tf"].values - pred_val_stribeck)

In [ ]:
model = Sequential([
    Dense(64, activation='tanh', input_shape=(4,)),
    Dense(64, activation='tanh'),
    Dense(32, activation='tanh'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
early_stop = EarlyStopping(patience = 20, restore_best_weights = True)
history = model.fit(
    X_train,
    residual_train,
    validation_data=(X_val, residual_val),
    epochs=1000,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
nn_residual = model.predict(X_test).flatten()

In [ ]:
pred3 = (pred2 + nn_residual)

In [ ]:
rmse3 = np.sqrt(mean_squared_error(tf_test, pred3))
mae3 = mean_absolute_error(tf_test, pred3)
r2_3 = r2_score(tf_test, pred3)

print(rmse3)
print(mae3)
print(r2_3)

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Coulomb + Viscous",
        "Stribeck",
        "GreyBox"
    ],
    "RMSE":[
        rmse1,
        rmse2,
        rmse3
    ],
    "MAE":[
        mae1,
        mae2,
        mae3
    ],
    "R2":[
        r2_1,
        r2_2,
        r2_3
    ]
})

results

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(tf_test, label='Actual')
plt.plot(pred3, label='GreyBox Prediction')
plt.legend()
plt.grid()
plt.show()

In [ ]:
error = tf_test - pred3
plt.figure(figsize=(15,5))
plt.plot(error)
plt.axhline(0, color='red')
plt.grid()
plt.show()

In [ ]:
idx = np.argsort(v_test)
plt.figure(figsize=(8,6))
plt.scatter(
    v_test, 
    tf_test, 
    s=2, 
    alpha=0.2
)
plt.plot(
    v_test[idx], 
    pred3[idx], 
    linewidth=3
)
plt.xlabel("Velocity")
plt.ylabel("Torque")
plt.show()